In [2]:
import pandas as pd
df = pd.read_csv(
    r"C:\Users\GIFTY\Projects\customer-analytics-portfolio\customer-lifetime-value-analysis\data\raw\online_retail_raw.csv",
    encoding="latin1"
)
df.head(3)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom


In [4]:
df['Revenue'] = df['Quantity'] * df['UnitPrice']


In [5]:
# Drop rows without customer
df = df.dropna(subset=['CustomerID'])

# Keep only positive revenue (NO returns in CLV)
df = df[df['Revenue'] > 0]

# Convert date
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])


In [6]:
df[['CustomerID', 'InvoiceDate', 'Revenue']].info()


<class 'pandas.core.frame.DataFrame'>
Index: 397884 entries, 0 to 541908
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   CustomerID   397884 non-null  float64       
 1   InvoiceDate  397884 non-null  datetime64[ns]
 2   Revenue      397884 non-null  float64       
dtypes: datetime64[ns](1), float64(2)
memory usage: 12.1 MB


In [7]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
snapshot_date


Timestamp('2011-12-10 12:50:00')

In [8]:
clv_df = (
    df.groupby('CustomerID')
      .agg(
          frequency=('InvoiceNo', 'nunique'),
          recency=('InvoiceDate', lambda x: (x.max() - x.min()).days),
          T=('InvoiceDate', lambda x: (snapshot_date - x.min()).days),
          monetary_value=('Revenue', 'mean')
      )
      .reset_index()
)


In [9]:
clv_df = clv_df[
    (clv_df['frequency'] > 0) &
    (clv_df['monetary_value'] > 0)
]


In [10]:
(clv_df['recency'] > clv_df['T']).sum()


0

#### GAMMA-GAMMA DATASET

In [11]:
gg_df = clv_df[clv_df['frequency'] > 1]


In [12]:
clv_df.describe()


,CustomerID,frequency,recency,T,monetary_value
count,4338.000000,4338.000000,4338.000000,4338.000000,4338.000000
mean,15300.408022,4.272015,130.448594,223.308207,68.350506
std,1721.808492,7.697998,132.039554,117.886471,1467.918896
min,12346.000000,1.000000,0.000000,1.000000,2.101286
25%,13813.250000,1.000000,0.000000,113.000000,12.365367
50%,15299.500000,2.000000,92.500000,249.000000,17.723119
75%,16778.750000,5.000000,251.750000,327.000000,24.858417
max,18287.000000,209.000000,373.000000,374.000000,77183.600000


In [13]:
clv_df.to_csv(
    "../data/processed/clv_modeling_dataset.csv",
    index=False
)
